In [35]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import anthropic
import copy
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [4]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [5]:
# Generate a random choice (True for right_answer, False for hallucinated_answer)
choice = np.random.rand(len(qa_data)) < 0.5

# Assign the chosen answer
qa_data["selected_answer"] = np.where(choice, qa_data["right_answer"], qa_data["hallucinated_answer"])

# Add a column indicating the source of the answer
qa_data["hallucinated_flag"] = np.where(choice, 0, 1)

In [6]:
qa_data_df = pd.DataFrame({
    "reference": qa_data.knowledge,
    "query": qa_data.question,
    "response": qa_data.selected_answer
})

In [16]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [15]:
import nest_asyncio
import os
from phoenix.evals import HallucinationEvaluator, run_evals, AnthropicModel

nest_asyncio.apply()  # This is needed for concurrency in notebook environments

# Get Anthropic API key from environment variable
anthropic_api_key = os.environ.get("ANTHROPIC_KEY")

# Set up Claude model for evaluation with API key
eval_model = AnthropicModel(
    model="claude-3-7-sonnet-20250219"
)

# Define your evaluators
hallucination_evaluator = HallucinationEvaluator(eval_model)

In [18]:
import nest_asyncio
import os
from phoenix.evals import HallucinationEvaluator, run_evals, AnthropicModel

nest_asyncio.apply()  # This is needed for concurrency in notebook environments

# Get Anthropic API key from environment variable
# anthropic_api_key = os.environ.get("ANTHROPIC_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = anthropic_key

# Set up Claude model for evaluation with API key
eval_model = AnthropicModel(
    model="claude-3-7-sonnet-20250219"
)

# Define your evaluators
hallucination_evaluator = HallucinationEvaluator(eval_model)

# We have to make some minor changes to our dataframe to use the column names expected by our evaluators
# for `hallucination_evaluator` the input df needs to have columns 'output', 'input', 'context'
# for `qa_evaluator` the input df needs to have columns 'output', 'input', 'reference'
qa_data_df["context"] = qa_data_df["reference"]
qa_data_df.rename(columns={"query": "input", "response": "output"}, inplace=True)
assert all(column in qa_data_df.columns for column in ["output", "input", "context", "reference"])

# Run the evaluators, each evaluator will return a dataframe with evaluation results
# We upload the evaluation results to Phoenix in the next step
qa_hallucination_eval_df = run_evals(
    dataframe=qa_data_df, evaluators=[hallucination_evaluator], provide_explanation=True
)

run_evals |          | 0/10000 (0.0%) | ⏳ 06:04<? | ?it/s
run_evals |██████████| 10000/10000 (100.0%) | ⏳ 43:06<00:00 |  3.87it/s


In [23]:
qa_hallucination_eval_df[0].to_csv('../../results/Hallucination/Arize/qa_results.csv')

In [27]:
qa_data_eval_results = qa_hallucination_eval_df[0]

In [ ]:
# Copy dataset
qa_data_eval = copy.deepcopy(qa_data)

# Add hallucination information from runs
qa_data_eval['hallucination_score'] = qa_data_eval_results.score
qa_data_eval['hallucination_label'] = qa_data_eval_results.label

In [38]:
# Compute metrics
accuracy = accuracy_score(qa_data_eval["hallucinated_flag"], qa_data_eval["hallucination_score"])
precision = precision_score(qa_data_eval["hallucinated_flag"], qa_data_eval["hallucination_score"])
recall = recall_score(qa_data_eval["hallucinated_flag"], qa_data_eval["hallucination_score"])

# Print results
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}%")


Accuracy: 84.79%
Precision: 85.40%
Recall: 84.47%
